# Group001 G0 — Shawn

1. canonical order count / 规范订单数量；
2. within-file duplicate `order_id` count / 文件内部重复 `order_id` 数量；
3. cross-file overlap `order_id` count / 文件之间重叠 `order_id` 数量。



## 0. Paths and environment / 路径与环境


In [ ]:
from pathlib import Path
import json
from collections import Counter

from bs4 import BeautifulSoup

GROUP_ID = "001"
GROUP_ALIAS = f"Group{GROUP_ID}"
JSON_NAME = f"{GROUP_ALIAS}_commerce.json"
XML_NAME = f"{GROUP_ALIAS}_operations.xml"

# Shared Colab folder, when available.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass


def locate_input_dir():
    candidates = [
        Path("/content/drive/MyDrive/Group001_A1/DATA/Group001_A1/raw_input"),
        Path("Group001_A1/DATA/Group001_A1/raw_input"),
        Path("DATA/Group001_A1/raw_input"),
        Path("raw_input"),
    ]
    for candidate in candidates:
        if (candidate / JSON_NAME).is_file() and (candidate / XML_NAME).is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate Group001 raw_input containing both source files."
    )


INPUT_DIR = locate_input_dir()
JSON_PATH = INPUT_DIR / JSON_NAME
XML_PATH = INPUT_DIR / XML_NAME

print("JSON:", JSON_PATH)
print("XML :", XML_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
JSON: /content/drive/MyDrive/Group001_A1/DATA/Group001_A1/raw_input/Group001_commerce.json
XML : /content/drive/MyDrive/Group001_A1/DATA/Group001_A1/raw_input/Group001_operations.xml


## 1. Read the two source files

In [ ]:
with JSON_PATH.open(encoding="utf-8") as file:
    commerce = json.load(file)

with XML_PATH.open(encoding="utf-8") as file:
    operations_soup = BeautifulSoup(file, "lxml-xml")

operations_root = operations_soup.find("OperationsExport")

print("JSON top-level keys:", list(commerce.keys()))
print("JSON orders list length:", len(commerce["orders"]))
print("XML root:", operations_root.name, dict(operations_root.attrs))

print("XML direct blocks:")
for block in operations_root.find_all(recursive=False):
    children = block.find_all(recursive=False)
    first_child = children[0].name if children else "-"
    print(f"  {block.name:20s} {len(children):5d} x <{first_child}>")

JSON top-level keys: ['customerProfiles', 'exportMetadata', 'orders', 'productReviews']
JSON orders list length: 2818
XML root: OperationsExport {'groupAlias': 'Group001', 'sourceSystem': 'OperationsERP', 'period': '2018'}
XML direct blocks:
  Export_Metadata          2 x <System>
  Orders                2818 x <Order>
  ProductCatalogue      1000 x <Product>
  ProductReviews        3946 x <Review>
  WarehouseDirectory       3 x <Warehouse>


## 2. Extract order IDs / 提取订单 ID


In [ ]:
# JSON order IDs, retaining source duplicates.
json_order_ids = [
    order["header"]["orderID"]
    for order in commerce["orders"]
]

# XML order IDs, using BeautifulSoup find/find_all.
xml_order_ids = []
for order in operations_root.find_all("Order"):
    header = order.find("Header", recursive=False)
    order_id_tag = (
        header.find("Order_ID", recursive=False)
        if header is not None
        else None
    )
    if order_id_tag is not None:
        order_id = order_id_tag.get_text(strip=True)
        if order_id:
            xml_order_ids.append(order_id)

print("JSON order records including duplicates:", len(json_order_ids))
print("XML order records including duplicates :", len(xml_order_ids))

JSON order records including duplicates: 2818
XML order records including duplicates : 2818


## 3. Within-file duplicate order IDs / 文件内部重复订单 ID


In [ ]:
json_order_counts = Counter(json_order_ids)
xml_order_counts = Counter(xml_order_ids)

json_within_duplicate_count = sum(
    1 for order_id, count in json_order_counts.items()
    if count > 1
)
xml_within_duplicate_count = sum(
    1 for order_id, count in xml_order_counts.items()
    if count > 1
)

print("JSON duplicated order_id values:", json_within_duplicate_count)
print("XML duplicated order_id values :", xml_within_duplicate_count)

if json_within_duplicate_count != xml_within_duplicate_count:
    print("Note: the two sources have different within-file duplicate counts.")

WITHIN_FILE_DUPLICATE_COUNT = json_within_duplicate_count

JSON duplicated order_id values: 68
XML duplicated order_id values : 68


## 4. Cross-file overlap / 文件之间的订单重叠

In [ ]:
json_unique_order_ids = set(json_order_ids)
xml_unique_order_ids = set(xml_order_ids)

overlap_order_ids = json_unique_order_ids & xml_unique_order_ids
CROSS_FILE_OVERLAP_COUNT = len(overlap_order_ids)

print("JSON distinct order IDs:", len(json_unique_order_ids))
print("XML distinct order IDs :", len(xml_unique_order_ids))
print("Cross-file overlap count:", CROSS_FILE_OVERLAP_COUNT)

JSON distinct order IDs: 2750
XML distinct order IDs : 2750
Cross-file overlap count: 500


## 5. Canonical order count / 规范订单数量


In [ ]:
canonical_order_ids = json_unique_order_ids | xml_unique_order_ids
CANONICAL_ORDER_COUNT = len(canonical_order_ids)

inclusion_exclusion_check = (
    len(json_unique_order_ids)
    + len(xml_unique_order_ids)
    - CROSS_FILE_OVERLAP_COUNT
)

assert inclusion_exclusion_check == CANONICAL_ORDER_COUNT

print("Canonical order count:", CANONICAL_ORDER_COUNT)
print(
    f"{len(json_unique_order_ids)} + {len(xml_unique_order_ids)} "
    f"- {CROSS_FILE_OVERLAP_COUNT} = {CANONICAL_ORDER_COUNT}"
)

Canonical order count: 5000
2750 + 2750 - 500 = 5000
